© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
%cd drive/MyDrive/project/semantic-segmentation-roads/eomt

/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/semantic-segmentation-roads/eomt


In [ ]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 76.9 MB/s e

## Setup

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = 0
img_idx = 10
data_path = "/content/drive/MyDrive/project/"
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [ ]:
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

## Load model

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")

if name is None:
    warnings.warn("No logger name found in the config. Please specify a model name.")
else:
    try:
        state_dict_path = "/content/drive/MyDrive/project/eomt_cityscapes.bin"

        is_dinov3 = "dinov3" in name

        if is_dinov3:
            model_kwargs["ckpt_path"] = state_dict_path
            model_kwargs["delta_weights"] = True

        model = (
            lit_cls(
                img_size=data.img_size,
                num_classes=data.num_classes,
                network=network,
                **model_kwargs,
            )
            .eval()
            .to(device)
        )

        if not is_dinov3:
            state_dict = torch.load(
                state_dict_path, map_location=f"cuda:{device}", weights_only=True
            )
            model.load_state_dict(state_dict, strict=False)

    except RepositoryNotFoundError:
        warnings.warn(
            f"Pre-trained model not found for `{name}`. Please load your own checkpoint."
        )

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data.img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")

    axes[1].imshow(pred_array)
    axes[1].set_title("Prediction")

    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    # Overlay numeri sulla prediction
    h, w = pred_array.shape
    step = 35  # ogni 10 pixel
    for i in range(0, h, step):
        for j in range(0, w, step):
          axes[1].text(j, i, str(int(pred_array[i, j])),
                      ha='center', va='center', color='white', fontsize=6)
    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# for img_idx in range(30, 60):
#   img, target = data.val_dataloader().dataset[img_idx]
#   pred_array, target_array = infer_semantic(img, target)
#   plot_semantic_results(img, pred_array, target_array)



In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex

iou_metric = MulticlassJaccardIndex(
    num_classes=data.num_classes,
    ignore_index=IGNORE_INDEX,
    average="macro"
)
iou_score = iou_metric(torch.tensor(pred_array), torch.tensor(target_array))
print(f"Image mIoU: {iou_score.item() * 100:.2f}%")

Image mIoU: 76.59%


In [ ]:
map_label_idx = {
    0: 'road',
    13: 'car',
    12: 'rider',
    11: 'person',
    7: 'traffic sign',
    6: 'traffic light',
    8: 'vegetation',
    1: 'sidewalk',
    14: 'truck',
    15: 'bus',
    5: 'pole',
    10: 'sky',
    2: 'building',
    18: 'bicycle',
    17: 'motorcycle',
    9: 'terrain',
    16: 'on rails',
    4: 'fence',
    3: 'wall'
}

In [ ]:
# map_label_idx = ['road', 'sidewalk', 'person', 'rider', 'car', 'truck', 'bus', 'on rails', 'motorcycle', 'bicycle', 'building', 'wall', 'fence', 'pole', 'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky']

In [ ]:
for img_idx in range(0, len(data.val_dataloader().dataset)):
  img, target = data.val_dataloader().dataset[img_idx]
  pred_array, target_array = infer_semantic(img, target)
  if pred_array.shape != (1024, 2048):
    print(pred_array.shape)
  for val in np.unique(pred_array):
    if val not in [0,  1,  2, 3, 4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18]:
      print(f'{img_idx} {np.unique(pred_array)} (val: {val})')

In [ ]:
from sklearn.metrics import confusion_matrix

def compute_iou(prediction, target, num_classes=18, ignore_index=255):
    prediction = prediction.flatten()
    target = target.flatten()
    mask = (target != ignore_index)
    prediction = prediction[mask]
    target = target[mask]
    if len(target) == 0: return np.zeros(num_classes), 0.0
    cm = confusion_matrix(target, prediction, labels=np.arange(num_classes))
    intersection = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - intersection
    iou = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
    valid_classes = (union > 0)
    miou = np.mean(iou[valid_classes]) if np.any(valid_classes) else 0.0
    return iou, miou
# 1. Recupera i dati dal dataset usando l'indice corrente
img, target = data.val_dataloader().dataset[img_idx]

# 2. Prepara l'immagine per il modello (aggiungi la dimensione del batch e sposta su GPU)
# Nota: usa il nome della variabile del tuo modello, qui assumiamo si chiami 'model'
img_input = img.unsqueeze(0).to(device)

# 3. Inferenza
with torch.no_grad():
    output = model(img_input)
    # Prendiamo l'argmax per avere la mappa dei segmenti (18 classi)
    semantic_prediction = torch.argmax(output, dim=1).squeeze().cpu().numpy()

# 4. Gestione Ground Truth
# Se il target è un dizionario (come nel panoptic)
if isinstance(target, dict):
    masks = target['masks'].numpy()
    labels = target['labels'].numpy()
    _, h, w = masks.shape
    gt_mask = np.full((h, w), 255, dtype=np.uint8)
    for mask, label in zip(masks, labels):
        gt_mask[mask.astype(bool)] = label
else:
    # Se il dataloader ti dà già la maschera 2D
    gt_mask = target.numpy()

# 5. Calcolo IoU
iou_per_class, miou = compute_iou(semantic_prediction, gt_mask)

print(f"--- Risultati Immagine {img_idx} ---")
print(f"mIoU: {miou:.4f}")

NameError: name 'img_tensor' is not defined

In [ ]:
from tqdm import tqdm
import numpy as np

# Inizializziamo la matrice di confusione globale (18x18 per Cityscapes)
global_cm = np.zeros((18, 18), dtype=np.int64)

# Prendiamo il dataloader di validazione
val_loader = data.val_dataloader()

print("Inizio valutazione globale...")
for batch in tqdm(val_loader):
    imgs, targets = batch
    imgs = imgs.to(device)

    # 1. Inferenza del modello (già addestrato su Cityscapes)
    with torch.no_grad():
        output = model(imgs)
        # Otteniamo la predizione semantica (Argmax)
        preds = torch.argmax(output, dim=1).cpu().numpy()

    # 2. Gestione Ground Truth
    # Nel notebook semantic, di solito target è un tensore (Batch, H, W)
    if isinstance(targets, dict):
        # Se il target è un dizionario (formato panoptic) lo ricostruiamo
        masks = targets['masks'].numpy()
        labels = targets['labels'].numpy()
        gt_masks = np.full(preds.shape, 255, dtype=np.uint8)
        # Nota: questo loop serve se il batch_size è 1
        for m, l in zip(masks, labels):
            gt_masks[0, m.astype(bool)] = l
    else:
        gt_masks = targets.numpy()

    # 3. Aggiornamento Matrice di Confusione
    # Filtriamo i pixel da ignorare (255)
    p_flat = preds.flatten()
    t_flat = gt_masks.flatten()
    mask = (t_flat != 255)

    global_cm += confusion_matrix(t_flat[mask], p_flat[mask], labels=np.arange(18))

# 4. Calcolo Finale IoU
intersection = np.diag(global_cm)
union = global_cm.sum(axis=1) + global_cm.sum(axis=0) - intersection
iou_per_class = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)

# Media solo sulle classi presenti nel dataset
valid_classes = (union > 0)
final_miou = np.mean(iou_per_class[valid_classes])

# 5. Visualizzazione finale
print(f"\n--- RISULTATI GLOBALI DATASET ---")
print(f"mIoU Totale: {final_miou:.4f}")
for i, name in enumerate(class_names):
    if union[i] > 0:
        print(f"{name:<15}: {iou_per_class[i]:.4f}")

Inizio valutazione globale...


  0%|          | 0/500 [00:01<?, ?it/s]


AttributeError: 'list' object has no attribute 'to'